In [1]:
import pandas as pd
import torch
print("GPU:",torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4060 Ti


In [2]:
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
dataset = pd.get_dummies(dataset, dtype=int, drop_first=True)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [4]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [5]:
indep = dataset[['Age', 'EstimatedSalary',  'Gender_Male']]
depend = dataset[['Purchased']]

In [6]:
depend.value_counts()

Purchased
0            257
1            143
Name: count, dtype: int64

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(indep,depend,test_size=0.30,random_state=0)

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

dt_param_grid = {
    "criterion": ["gini", "entropy", "log_loss"],
    "splitter": ["best", "random"],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [None, "sqrt", "log2"]
}



grid = GridSearchCV(DecisionTreeClassifier(), dt_param_grid, refit=True ,n_jobs=3, scoring='f1_weighted')
grid.fit(X_train,y_train)


,estimator,DecisionTreeClassifier()
,param_grid,"{'criterion': ['gini', 'entropy', ...], 'max_depth': [None, 5, ...], 'max_features': [None, 'sqrt', ...], 'min_samples_leaf': [1, 2, ...], ...}"
,scoring,'f1_weighted'
,n_jobs,3
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [10]:
re = grid.cv_results_
print("The classification value for best parameter {}:".format(grid.best_params_))
table = pd.DataFrame.from_dict(re)
table

The classification value for best parameter {'criterion': 'gini', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 10, 'splitter': 'random'}:


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,param_max_features,param_min_samples_leaf,param_min_samples_split,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002885,0.000568,0.005086,0.000572,gini,None,None,1,2,best,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.799537,0.821429,0.804584,0.875548,0.891667,0.838553,0.037839,515
1,0.002301,0.000401,0.004503,0.000551,gini,None,None,1,2,random,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.819142,0.819142,0.770909,0.816856,0.853485,0.815907,0.026301,649
2,0.001702,0.000602,0.005201,0.000247,gini,None,None,1,5,best,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.799537,0.840114,0.804584,0.929144,0.909115,0.856499,0.053393,290
3,0.001827,0.000212,0.004802,0.000511,gini,None,None,1,5,random,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.802399,0.819142,0.858503,0.809578,0.870721,0.832069,0.027372,554
4,0.002100,0.000201,0.005001,0.000316,gini,None,None,1,10,best,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.819142,0.840114,0.859435,0.911105,0.909115,0.867782,0.036841,147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,0.001800,0.000400,0.004502,0.000316,log_loss,30,log2,4,2,random,"{'criterion': 'log_loss', 'max_depth': 30, 'ma...",0.779334,0.567730,0.604980,0.841045,0.888158,0.736249,0.127707,790
806,0.001702,0.000401,0.004601,0.000374,log_loss,30,log2,4,5,best,"{'criterion': 'log_loss', 'max_depth': 30, 'ma...",0.855314,0.858503,0.876643,0.855556,0.926743,0.874552,0.027264,75
807,0.001701,0.000400,0.004544,0.000406,log_loss,30,log2,4,5,random,"{'criterion': 'log_loss', 'max_depth': 30, 'ma...",0.503106,0.799537,0.769053,0.874356,0.652952,0.719801,0.129689,799
808,0.002201,0.000509,0.004802,0.000511,log_loss,30,log2,4,10,best,"{'criterion': 'log_loss', 'max_depth': 30, 'ma...",0.874254,0.819142,0.841398,0.833784,0.927778,0.859271,0.038722,237


In [11]:
y_pred = grid.predict(X_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 0, 0], dtype=int64)

In [12]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_pred,y_test)
print(cm)

[[74  7]
 [ 5 34]]


In [13]:
from sklearn.metrics import classification_report
report= classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.91      0.94      0.93        79
           1       0.87      0.83      0.85        41

    accuracy                           0.90       120
   macro avg       0.89      0.88      0.89       120
weighted avg       0.90      0.90      0.90       120

